# Hierarchical Multi-Agent Deep Research & Report Planner
### Pattern: Supervisor Orchestrator with Parallel Sub-Agent Delegation (Scatter-Gather / Map-Reduce)

This notebook demonstrates a **hierarchical multi-agent architecture** built with **LangChain (`create_agent`)** and **Google Gemini**.
It implements an autonomous deep research pipeline that breaks complex research requests into parallel sub-tasks, conducts real-time web research, synthesizes findings, and compiles a comprehensive Wikipedia-style article complete with an infobox, dynamic Table of Contents, and citations.

---

### Why Hierarchical Multi-Agent Architecture for Deep Research?
1. **Context Window Isolation**: Conducting in-depth web searches generates thousands of tokens of search snippets. By isolating each section's research inside a dedicated subagent, the main coordinator's context stays clean and focused on high-level orchestration.
2. **Parallel Concurrency (Scatter-Gather)**: Rather than researching sections sequentially, the coordinator triggers multiple tool calls concurrently in a single step. All body sections are researched simultaneously, drastically reducing overall latency.
3. **Specialized Prompts & Personas**: Writing an introduction requires broad conceptual framing; writing a conclusion requires synthesizing insights and structuring a summary table; writing body sections requires factual technical depth; and the compiler enforces strict Wikipedia formatting rules.
4. **Tool Partitioning**: Only the `Section_Researcher_Writer` agent receives web search tools. The `Intro_Writer`, `Conclusion_Writer`, and `Report_Compiler` are pure synthesis agents, preventing unnecessary API queries or search hallucination.

```
                               ┌─────────────────────────┐
                               │       User Request      │
                               │  (Deep Research Topic)  │
                               └────────────┬────────────┘
                                            │
                                            ▼
                               ┌─────────────────────────┐
                               │ Main Coordinator Agent  │
                               │ (Deep Research Planner) │
                               └────────────┬────────────┘
                                            │
           ┌────────────────────────────────┼────────────────────────────────┐
           │ [PARALLEL FAN-OUT 1]           │ [PARALLEL FAN-OUT 2]           │ [FAN-IN SYNTHESIS]
           ▼                                ▼                                ▼
┌──────────────────────┐         ┌──────────────────────┐         ┌──────────────────────┐
│   Section Writers    │         │  Intro & Conclusion  │         │   Report Compiler    │
│ (Parallel Subagents) │         │ (Parallel Subagents) │         │ (Wikipedia Styler)   │
│                      │         │                      │         │                      │
│  ┌────────────────┐  │         │  ┌────────────────┐  │         │  ┌────────────────┐  │
│  │Section 1 Writer│  │         │  │  Intro Writer  │  │         │  │ Lead Paragraph │  │
│  ├────────────────┤  │         │  ├────────────────┤  │         │  ├────────────────┤  │
│  │Section 2 Writer│  │         │  │Conclusion Writ.│  │         │  │Infobox & Headg.│  │
│  ├────────────────┤  │         │  └────────────────┘  │         │  ├────────────────┤  │
│  │Section 3 Writer│  │         │                      │         │  │Auto TOC / File │  │
│  └───────┬────────┘  │         └──────────────────────┘         │  │  Save to Disk  │  │
└──────────┼───────────┘                                          └──────────────────────┘
           │
           ▼
   [Tavily Search API]
```

## 1. Setup & Environment Variables
In this section, we load environment variables from the `.env` file or prompt interactively.
We need two primary API keys:
- `GEMINI_API_KEY` (or `GOOGLE_API_KEY`): Powers Google Gemini 2.5/3.x models for agent reasoning and report synthesis.
- `TAVILY_API_KEY`: Powers real-time search extraction optimized for LLMs (extracts clean text rather than noisy HTML scraping).

In [1]:
import os
from getpass import getpass
from dotenv import load_dotenv

# 1. Load keys from root directory or parent directory .env if present
if os.path.exists(".env"):
    load_dotenv(dotenv_path=".env")
elif os.path.exists("../.env"):
    load_dotenv(dotenv_path="../.env")
else:
    load_dotenv()

# 2. Configure Google Gemini API Key (supports either GEMINI_API_KEY or GOOGLE_API_KEY)
if "GEMINI_API_KEY" not in os.environ:
    if "GOOGLE_API_KEY" in os.environ:
        os.environ["GEMINI_API_KEY"] = os.environ["GOOGLE_API_KEY"]
    else:
        os.environ["GEMINI_API_KEY"] = getpass("Enter your GEMINI API Key: ")

# 3. Configure Tavily Search API Key for real-time web grounding
if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = getpass("Enter your Tavily API Key: ")

print("✓ API credentials loaded and verified successfully!")

In [2]:
# Prompt Inspector auto-patches client calls to provide real-time visibility
# into outgoing LLM system prompts, user messages, and tool invocations.
import prompt_inspector
prompt_inspector.inspect()

INFO:prompt-inspector:OpenAI client auto-patched successfully!
INFO:prompt-inspector:Anthropic client auto-patched successfully!
INFO:prompt-inspector:google-genai auto-patched successfully!


## 2. Research Tools Definition
We define the `search_web` tool using LangChain's `@tool` decorator and `TavilySearchAPIWrapper`.

### Design Considerations:
- **Curated Results (`max_results=3`)**: Capping the top results maintains a high signal-to-noise ratio and prevents context window saturation.
- **Clean Attribution**: Results are formatted with explicit `Source`, `URL`, and clean `Content` so that downstream writer agents can cite their sources faithfully.
- **Defensive Exception Handling**: Any network error or rate limit is caught and returned as a graceful string rather than crashing the agent execution loop.

In [3]:
from langchain.tools import tool
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper

# Initialize Tavily wrapper (reads TAVILY_API_KEY automatically from environment)
tavily_wrapper = TavilySearchAPIWrapper()

@tool
def search_web(query: str) -> str:
    """Searches the web for up-to-date information on a query.
    
    Args:
        query (str): The search query string.
        
    Returns:
        str: Formatted markdown string containing source titles, URLs, and snippet contents.
    """
    try:
        # Query Tavily with a focused result count (top 3 sources per section)
        results = tavily_wrapper.results(query, max_results=3)
        if not results:
            return f"No results found for query: '{query}'."
        
        # Format clean, structured output with source attribution
        output = f"Web Search Results for '{query}':\n"
        for i, res in enumerate(results, 1):
            output += f"Source {i}: {res.get('title', 'Untitled')}\n"
            output += f"URL: {res.get('url', 'N/A')}\n"
            output += f"Content: {res.get('content', '')}\n\n"
        return output
    except Exception as e:
        # Graceful fallback: return error string so agent can re-try or reason
        return f"Web search encountered an error: {str(e)}"

## 3. Specialized Subagent Instantiation
We create four specialized subagents using LangChain's `create_agent`.
Notice the **strict separation of concerns**:

| Subagent | Role | Tools | System Prompt Focus |
| :--- | :--- | :--- | :--- |
| **1. Section Researcher & Writer** | Technical Section Author | `[search_web]` | Conducts live web research, extracts facts, writes 150-200 words with citations. |
| **2. Introduction Writer** | Narrative Opening Author | `[]` (None) | Reads completed body sections and frames the overarching problem/context. |
| **3. Conclusion Writer** | Synthesis & Outlook Author | `[]` (None) | Synthesizes findings across body sections and constructs a Markdown summary table. |
| **4. Report Compiler** | Wikipedia Formatter | `[]` (None) | Formats the final document with an Infobox table, lead paragraph, clean heading hierarchy, and References. |

In [4]:
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

# Base LLM: gemini-3.1-flash-lite provides rapid execution and high reasoning quality at low latency
# Temperature=0 ensures deterministic, factual responses across all research agents
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)

# ----------------------------------------------------------------------------------------
# Subagent 1: Section Researcher & Writer (Equipped with web search tool)
# ----------------------------------------------------------------------------------------
section_writer_agent = create_agent(
    model=llm,
    tools=[search_web],
    system_prompt=(
        "You are a specialized Section Researcher & Writer Agent. Your job is to research "
        "and write a detailed section of a report in Markdown based on a given topic, section name, "
        "and description. Use the search_web tool to gather information. "
        "Write a highly technical, factual section of about 150-200 words. "
        "Start with a bold key insight, use short paragraphs, and list your sources at the end."
    )
)

# ----------------------------------------------------------------------------------------
# Subagent 2: Intro Writer (Pure synthesis agent - no tools needed)
# ----------------------------------------------------------------------------------------
intro_writer_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "You are a specialized Introduction Writer Agent. Your job is to read "
        "completed body sections of a report and write a compelling Introduction "
        "(50-100 words, setting context). Do not call any tools, write directly based on the provided section drafts."
    )
)

# ----------------------------------------------------------------------------------------
# Subagent 3: Conclusion Writer (Pure synthesis agent - produces summary table)
# ----------------------------------------------------------------------------------------
conclusion_writer_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "You are a specialized Conclusion Writer Agent. Your job is to read "
        "completed body sections of a report and write a structured Conclusion "
        "(100-150 words, synthesizing key findings and ending with next steps). "
        "Include a structured Markdown table summarizing the insights. Do not call any tools."
    )
)

# ----------------------------------------------------------------------------------------
# Subagent 4: Report Compiler (Wikipedia article formatting & infobox assembly)
# ----------------------------------------------------------------------------------------
report_compiler_agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "You are a specialized Report Compiler Agent. Your job is to assemble all sections "
        "(Introduction, Body Sections, and Conclusion) into a single cohesive Wikipedia-style Markdown article.\n"
        "Structure it like a Wikipedia entry:\n"
        "1. Begin with a clean top-level heading (# [Topic]).\n"
        "2. Place a Wikipedia-style Infobox (as a Markdown table with bold keys) right at the beginning.\n"
        "3. Write a clear, concise introductory lead paragraph summarizing the topic before any subheadings.\n"
        "4. Do NOT write a Table of Contents manually; a python script will automatically inject it after the lead paragraph.\n"
        "5. Use clear, structured subheadings (## [Section] and ### [Subsection]).\n"
        "6. Format code terms using inline backticks.\n"
        "7. End with a structured 'References' section listing all sources cleanly.\n"
        "Ensure all special characters like '$' are escaped as '\\$' for correct rendering."
    )
)

print("✓ All 4 specialized subagents created successfully!")

## 4. Wrapping Subagents as Tools (The Subagents Pattern)
To enable the coordinator to delegate tasks, we wrap each subagent inside a LangChain `@tool` function.

### Key Mechanics to Understand:
1. **Isolated Invocation (`subagent.invoke(...)`)**:
   Each subagent receives its own isolated messages list. The intermediate tool iterations (e.g. searching Tavily, reading sources) remain inside the subagent's scratchpad and **do not leak** into the parent agent's context.
2. **Callback Propagation (`RunnableConfig`)**:
   By passing the `config: RunnableConfig` parameter to `subagent.invoke(...)`, telemetry, logging, and LangSmith tracing from the subagent bubble up to the parent execution graph.
3. **Timestamp Instrumentation**:
   We print microsecond timestamps (`[HH:MM:SS.mmm] >>> ENTER / <<< EXIT`) upon tool invocation. When the coordinator invokes multiple subagents concurrently, these timestamps prove that tools execute in parallel!
4. **Automated Table of Contents & Disk Persistence (`save_wikipedia_style_report`)**:
   A dedicated export tool parses all H2 and H3 markdown headings, dynamically generates clickable Markdown anchor links, inserts the Table of Contents right after the lead paragraph, and writes the final `.md` file to the `reports/` folder.

In [5]:
from langchain_core.runnables import RunnableConfig
import datetime
import re
import os

# ========================================================================================
# Utility: Save Wikipedia-Style Report with Dynamic Table of Contents
# ========================================================================================
def save_wikipedia_style_report(report_title: str, report_content: str):
    """Cleans, formats, generates an automatic Table of Contents, and saves the report to disk.
    
    Args:
        report_title (str): Title of the research topic.
        report_content (str): Full Markdown text compiled from all sections.
        
    Returns:
        tuple[str, str]: (final_content_with_toc, saved_report_file_path)
    """
    # 1. Sanitize report title to create a safe, clean file name
    safe_title = re.sub(r'[^a-zA-Z0-9\s\_\-]', '', report_title)
    safe_title = re.sub(r'\s+', '_', safe_title.strip())
    
    cleaned_content = report_content.strip()
    
    # 2. Normalize the top-level H1 title to '# [report_title]'
    if cleaned_content.startswith('#'):
        lines = cleaned_content.split('\n')
        orig_title = lines[0].replace('#', '').strip()
        if orig_title.lower().startswith('report:'):
            orig_title = orig_title[7:].strip()
        lines[0] = f'# {orig_title if orig_title else report_title}'
        cleaned_content = '\n'.join(lines)
    else:
        cleaned_content = f'# {report_title}\n\n{cleaned_content}'
        
    # 3. Parse all H2 and H3 markdown headings using regex
    headings = re.findall(r'^(#{2,3})\s+(.*)$', cleaned_content, re.MULTILINE)
    toc_lines = ['## Contents\n']
    for level_hashes, title in headings:
        # Indent subheadings (H3) by 2 spaces under H2
        indent = '  ' * (len(level_hashes) - 2)
        clean_title = re.sub(r'[\*\`\_]', '', title)
        # Generate clean URL anchor slug (e.g., 'Theoretical Foundations' -> 'theoretical-foundations')
        anchor = clean_title.lower()
        anchor = re.sub(r'[^a-z0-9\s\-]', '', anchor)
        anchor = re.sub(r'\s+', '-', anchor)
        toc_lines.append(f'{indent}- [{clean_title}](#{anchor})')
    
    toc_str = '\n'.join(toc_lines) + '\n\n---\n'
    
    # 4. Insert the Table of Contents before the first H2 subheading (after lead paragraph & infobox)
    first_h2_match = re.search(r'^##\s+', cleaned_content, re.MULTILINE)
    if first_h2_match:
        idx = first_h2_match.start()
        final_content = cleaned_content[:idx] + toc_str + cleaned_content[idx:]
    else:
        # Fallback: Prepend TOC right below the H1 title
        lines = cleaned_content.split('\n')
        insert_idx = 1
        while insert_idx < len(lines) and not lines[insert_idx].strip():
            insert_idx += 1
        lines.insert(insert_idx, '\n' + toc_str)
        final_content = '\n'.join(lines)
        
    # 5. Ensure reports destination directory exists
    os.makedirs('reports', exist_ok=True)
    report_path = f'reports/{safe_title}.md'
    
    # 6. Write final markdown to disk with UTF-8 encoding
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write(final_content)
        
    print(f'\n[Wikipedia-Style Export] Saved report to: {os.path.abspath(report_path)}')
    return final_content, report_path

# ========================================================================================
# Tool 1: Save Wikipedia-Style Report to Disk
# ========================================================================================
@tool("Save_Wikipedia_Style_Report", description="Saves the final compiled report in a Wikipedia-style Markdown format to disk. Input should include the report_title (e.g. 'LangGraph State Management') and the full report_content.")
def call_save_wikipedia_report(report_title: str, report_content: str) -> str:
    """Saves the final report to disk in a Wikipedia-style Markdown format."""
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] >>> ENTER: Save_Wikipedia_Style_Report (Title: {report_title})")
    try:
        wiki_content, report_path = save_wikipedia_style_report(report_title, report_content)
        now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
        print(f"\n[{now}] <<< EXIT: Save_Wikipedia_Style_Report finished")
        return f"Success: Wikipedia-style report saved successfully to {report_path}."
    except Exception as e:
        now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
        print(f"\n[{now}] <<< EXIT: Save_Wikipedia_Style_Report failed: {str(e)}")
        return f"Error saving report: {str(e)}"

# ========================================================================================
# Tool 2: Call Section Researcher & Writer Subagent
# ========================================================================================
@tool("Section_Researcher_Writer", description="Researches and writes a single section of a report. Input should be a query containing the section name, description, and overall topic.")
def call_section_writer(query: str, config: RunnableConfig) -> str:
    """Call the section researcher subagent. Runs isolated ReAct search and writes section draft."""
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] >>> ENTER: Section_Researcher_Writer (Query: {query[:60]}...)")
    # Invocation creates an isolated messages history; parent only receives final string content
    result = section_writer_agent.invoke({'messages': [{'role': 'user', 'content': query}]}, config)
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] <<< EXIT: Section_Researcher_Writer finished")
    return result['messages'][-1].content

# ========================================================================================
# Tool 3: Call Introduction Writer Subagent
# ========================================================================================
@tool("Intro_Writer", description="Drafts the Introduction section. Input must include the overall topic and all completed body sections as context.")
def call_intro_writer(query: str, config: RunnableConfig) -> str:
    """Call the intro subagent. Synthesizes body section drafts into a cohesive introduction."""
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] >>> ENTER: Intro_Writer")
    result = intro_writer_agent.invoke({'messages': [{'role': 'user', 'content': query}]}, config)
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] <<< EXIT: Intro_Writer finished")
    return result['messages'][-1].content

# ========================================================================================
# Tool 4: Call Conclusion Writer Subagent
# ========================================================================================
@tool("Conclusion_Writer", description="Drafts the Conclusion section. Input must include the overall topic and all completed body sections as context.")
def call_conclusion_writer(query: str, config: RunnableConfig) -> str:
    """Call the conclusion subagent. Summarizes key findings and generates an insights table."""
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] >>> ENTER: Conclusion_Writer")
    result = conclusion_writer_agent.invoke({'messages': [{'role': 'user', 'content': query}]}, config)
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] <<< EXIT: Conclusion_Writer finished")
    return result['messages'][-1].content

# ========================================================================================
# Tool 5: Call Report Compiler Subagent
# ========================================================================================
@tool("Report_Compiler", description="Assembles and compiles the final report structure. Input must contain the introduction, all body sections, and the conclusion to format and combine.")
def call_report_compiler(query: str, config: RunnableConfig) -> str:
    """Call the report compiler subagent. Formats sections into a cohesive Wikipedia-style Markdown article."""
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] >>> ENTER: Report_Compiler")
    result = report_compiler_agent.invoke({'messages': [{'role': 'user', 'content': query}]}, config)
    now = datetime.datetime.now().strftime("%H:%M:%S.%f")[:-3]
    print(f"\n[{now}] <<< EXIT: Report_Compiler finished")
    return result['messages'][-1].content

# Bundle tools for the Main Coordinator Agent
main_agent_tools = [
    call_section_writer, 
    call_intro_writer, 
    call_conclusion_writer, 
    call_report_compiler, 
    call_save_wikipedia_report
]

print(f"✓ Configured {len(main_agent_tools)} coordinator tools for delegation!")

## 5. Main Coordinator Agent (Deep Research Planner)
The Main Agent acts as the **executive research coordinator**.
Instead of doing web searches or draft writing directly, it plans the document structure and delegates to specialized subagents.

### How Parallel Execution is Achieved (Scatter-Gather / Map-Reduce):
1. **Multiple Tool Calls in One Turn**: Modern LLMs (like Gemini) support generating multiple tool calls in a single response turn.
2. **Parallel Step 1 (Scatter Body Sections)**: The coordinator generates tool calls for all 3 planned body sections simultaneously in turn 1. The runtime executes `call_section_writer` concurrently for all sections.
3. **Parallel Step 2 (Scatter Intro & Conclusion)**: Once the body sections return, the coordinator generates tool calls for `Intro_Writer` and `Conclusion_Writer` simultaneously in turn 2.
4. **Step 3 (Gather & Compile)**: The compiler combines all parts into a unified article.
5. **Step 4 (Persist & Deliver)**: The file saver writes the finished report to `reports/` and returns the final response to the user.

In [6]:
# System instructions explicitly enforcing parallel tool calls and strict workflow stages
main_agent_prompt = (
    "You are the Deep Research Planner, a main coordinator agent. "
    "Your goal is to coordinate a deep research report generation on a topic requested by the user. "
    "\n\n"
    "Available subagents and tools:\n"
    "1. Section_Researcher_Writer: Researches and writes a single section of the report.\n"
    "2. Intro_Writer: Drafts the Introduction using the body sections.\n"
    "3. Conclusion_Writer: Drafts the Conclusion using the body sections.\n"
    "4. Report_Compiler: Combines, formats, and compiles all sections into a single cohesive Markdown report.\n"
    "5. Save_Wikipedia_Style_Report: Saves the final compiled report in a Wikipedia-style Markdown format to disk.\n"
    "\n\n"
    "Step-by-step workflow:\n"
    "a. Determine the structure of the report (e.g., plan 2-3 main body sections based on the user's topic).\n"
    "b. Call Section_Researcher_Writer FOR ALL PLANNED SECTIONS IN PARALLEL (trigger multiple tool calls in a single turn/step) to write all body sections concurrently.\n"
    "c. Once the body sections are written, call BOTH Intro_Writer and Conclusion_Writer IN PARALLEL (trigger tool calls for both in a single turn/step) to write the introduction and conclusion concurrently.\n"
    "d. Call Report_Compiler with the Intro, the drafted sections, and the Conclusion to build the compiled report.\n"
    "e. Call Save_Wikipedia_Style_Report with the report title (e.g., 'LangGraph State Management') and the compiled report content to save it to disk.\n"
    "f. Output the final compiled report and the tool confirmation message to the user.\n\n"
    "CRITICAL: ALWAYS invoke parallel tool calls together in the same step to achieve concurrency. Do not call them one by one."
)

# Create the Main Coordinator Agent with our bundled delegation tools
main_agent = create_agent(
    model=llm,
    tools=main_agent_tools,
    system_prompt=main_agent_prompt
)

print("✓ Main Deep Research Coordinator Agent initialized successfully!")

## 6. Execution Run & Observability
We now test the end-to-end multi-agent system on a complex, multi-faceted scientific topic: **'Multiverse'**.

### What to Watch For in the Execution Trace Below:
- Notice the timestamps: `[20:37:15.863]`, `[20:37:15.865]`, and `[20:37:15.867]`. All three `Section_Researcher_Writer` calls start within **4 milliseconds** of each other, confirming true parallel execution!
- Notice the second parallel stage: `Intro_Writer` (`20:37:26.746`) and `Conclusion_Writer` (`20:37:26.747`) execute concurrently.
- Finally, `Report_Compiler` and `Save_Wikipedia_Style_Report` assemble and save the completed article.

In [7]:
# Define user research query requiring multi-perspective deep investigation
user_request = (
    "I want a research report on 'Multiverse'. "
    "What is the current state of research on the Multiverse? Include sections on theoretical foundations, "
    "observational evidence, and implications for physics and cosmology."
)

print("=== STARTING DEEP RESEARCH COORDINATOR AGENT ===\n")

# Invoke the Main Coordinator Agent
result = main_agent.invoke({"messages": [{"role": "user", "content": user_request}]})

# Extract and display the final synthesized report
print("\n=== FINAL REPORT RESPONSE ===\n")
final_response = result["messages"][-1].content
if isinstance(final_response, list):
    print(final_response[0]['text'])
else:
    print(final_response)

=== STARTING DEEP RESEARCH COORDINATOR AGENT ===\n


DEBUG wrapped_generate (new SDK):
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"
DEBUG wrapped_generate (new SDK):
DEBUG wrapped_generate (new SDK):
DEBUG wrapped_generate (new SDK):



[20:37:15.863] >>> ENTER: Section_Researcher_Writer (Query: Theoretical foundations of the Multiverse, including inflati...)

[20:37:15.865] >>> ENTER: Section_Researcher_Writer (Query: Observational evidence and challenges in detecting the Multi...)

[20:37:15.867] >>> ENTER: Section_Researcher_Writer (Query: Implications of the Multiverse for physics and cosmology, in...)


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"
DEBUG wrapped_generate (new SDK):
DEBUG wrapped_generate (new SDK):
INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"



[20:37:21.932] <<< EXIT: Section_Researcher_Writer finished


...


[20:37:36.294] <<< EXIT: Report_Compiler finished


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"
DEBUG wrapped_generate (new SDK):



[20:37:40.312] >>> ENTER: Save_Wikipedia_Style_Report (Title: Multiverse Research Report)

[Wikipedia-Style Export] Saved report to: /Users/sachinmishra/Desktop/Agents_From_Scratch/Langchain_Agents/reports/Multiverse_Research_Report.md

[20:37:40.316] <<< EXIT: Save_Wikipedia_Style_Report finished


INFO:httpx:HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent "HTTP/1.1 200 OK"


\n=== FINAL REPORT RESPONSE ===\n
The research report on the 'Multiverse' has been successfully compiled and saved. You can find the full report below:

# Multiverse

| Key | Information |
| :--- | :--- |
| **Concept** | Hypothetical set of multiple observable universes |
| **Primary Fields** | Cosmology, String Theory, Quantum Mechanics |
| **Key Mechanisms** | Eternal Inflation, String Landscape, Many-Worlds Interpretation |
| **Status** | Theoretical framework; currently unverified |

The concept of the multiverse challenges our fundamental understanding of reality, suggesting that our observable universe is merely one of countless distinct domains. This report explores the theoretical foundations of this hypothesis, drawing on inflation, string theory, and quantum mechanics to explain how such vast structures might arise. By examining potential observational evidence, including cosmic microwave background anomalies and theoretical bubble collisions, we investigate the validity of t